In [1]:
!pip install spacy scikit-learn nltk matplotlib pandas
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 41.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('vader_lexicon')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer

import spacy
nlp = spacy.load("en_core_web_sm")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries loaded successfully.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Libraries loaded successfully.


In [ ]:
## Step 1: Load Dataset
For this project, we will upload a CSV file into Google Colab.
The dataset should contain:
- a text column
- a category/label column

In [4]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
# Replace with your filename after upload
df = pd.read_csv("train.csv")  # Change this if your file name is different

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# Adjust column names if needed
# Example possibilities:
# text column: text, content, article, description
# category column: category, label, class

text_column = "text"       # change if needed
category_column = "category"  # change if needed

df = df[[text_column, category_column]].dropna()
df = df.rename(columns={text_column: "content", category_column: "category"})

print(df.head())
print(df["category"].value_counts())

In [ ]:
## Step 2: Explore the Dataset
We first examine category distribution and article length before preprocessing.

In [ ]:
df["text_length"] = df["content"].apply(lambda x: len(str(x).split()))

print("Dataset shape:", df.shape)
print("\nCategory counts:")
print(df["category"].value_counts())

In [ ]:
df["category"].value_counts().plot(kind="bar")
plt.title("Number of Articles by Category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [4]:
plt.hist(df["text_length"], bins=30)
plt.title("Distribution of Article Length")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
## Step 3: Text Preprocessing
We clean and normalize the text by:
- converting to lowercase
- removing punctuation and numbers
- removing stopwords
- lemmatizing words

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return " ".join(tokens)

df["clean_text"] = df["content"].apply(preprocess_text)

df[["content", "clean_text"]].head()

In [ ]:
print("Original article:")
print(df["content"].iloc[0][:500])

print("\nCleaned article:")
print(df["clean_text"].iloc[0][:500])

In [ ]:
## Step 4: TF-IDF Feature Extraction
TF-IDF converts text into numerical features so machine learning models can understand article content.

In [ ]:
tfidf = TfidfVectorizer(max_features=1500, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df["clean_text"])
feature_names = tfidf.get_feature_names_out()

print("TF-IDF matrix shape:", X_tfidf.shape)

In [ ]:
# Average TF-IDF score across all documents
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).flatten()
top_indices = mean_tfidf.argsort()[-15:][::-1]

top_words = [feature_names[i] for i in top_indices]
top_scores = [mean_tfidf[i] for i in top_indices]

plt.figure(figsize=(10,5))
plt.bar(top_words, top_scores)
plt.title("Top 15 TF-IDF Terms")
plt.xticks(rotation=45, ha='right')
plt.ylabel("Average TF-IDF Score")
plt.show()

In [ ]:
# Top terms by category
categories = df["category"].unique()

for cat in categories:
    cat_text = df[df["category"] == cat]["clean_text"]
    cat_matrix = tfidf.transform(cat_text)
    cat_mean = np.asarray(cat_matrix.mean(axis=0)).flatten()
    top_cat_idx = cat_mean.argsort()[-10:][::-1]
    top_cat_terms = [feature_names[i] for i in top_cat_idx]

    print(f"\nTop terms for {cat}:")
    print(top_cat_terms)

In [ ]:
## Step 5: Part-of-Speech (POS) Analysis
We use spaCy to identify nouns, verbs, adjectives, and other grammatical patterns in the articles.

In [ ]:
def get_pos_counts(text):
    doc = nlp(text[:1000000])  # prevent very large text issues
    pos_counts = {}
    for token in doc:
        pos = token.pos_
        pos_counts[pos] = pos_counts.get(pos, 0) + 1
    return pos_counts

In [ ]:
# POS analysis on a sample article
sample_doc = nlp(df["content"].iloc[0])

for token in sample_doc[:20]:
    print(token.text, "->", token.pos_)

In [ ]:
# Compare average noun/verb/adjective usage by category
pos_summary = []

for cat in df["category"].unique():
    subset = df[df["category"] == cat]["content"].head(20)  # small sample for speed
    noun_total, verb_total, adj_total = 0, 0, 0
    doc_count = 0

    for text in subset:
        doc = nlp(str(text))
        noun_total += sum(1 for token in doc if token.pos_ == "NOUN")
        verb_total += sum(1 for token in doc if token.pos_ == "VERB")
        adj_total += sum(1 for token in doc if token.pos_ == "ADJ")
        doc_count += 1

    pos_summary.append({
        "category": cat,
        "avg_nouns": noun_total / doc_count,
        "avg_verbs": verb_total / doc_count,
        "avg_adjectives": adj_total / doc_count
    })

pos_df = pd.DataFrame(pos_summary)
pos_df

In [ ]:
pos_df.plot(x="category", kind="bar", figsize=(10,5))
plt.title("Average POS Counts by Category")
plt.ylabel("Average Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
## Step 6: Syntax and Dependency Analysis
Dependency parsing helps us understand relationships between words, such as subjects, verbs, and objects.

In [ ]:
doc = nlp(df["content"].iloc[0])

for token in doc[:20]:
    print(f"Word: {token.text:<12} Dep: {token.dep_:<10} Head: {token.head.text}")

In [ ]:
# Count common dependency tags in a few sample articles
dep_counts = {}

for text in df["content"].head(20):
    doc = nlp(str(text))
    for token in doc:
        dep = token.dep_
        dep_counts[dep] = dep_counts.get(dep, 0) + 1

sorted_deps = sorted(dep_counts.items(), key=lambda x: x[1], reverse=True)[:10]
dep_labels = [x[0] for x in sorted_deps]
dep_values = [x[1] for x in sorted_deps]

plt.figure(figsize=(10,5))
plt.bar(dep_labels, dep_values)
plt.title("Top Dependency Relations")
plt.xlabel("Dependency Tag")
plt.ylabel("Count")
plt.show()

In [ ]:
## Step 7: Sentiment Analysis
We use VADER sentiment analysis to identify whether articles have positive, negative, or neutral tone.

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["sentiment"] = df["content"].apply(get_sentiment)
df[["content", "sentiment"]].head()

In [ ]:
print(df["sentiment"].value_counts())

df["sentiment"].value_counts().plot(kind="bar")
plt.title("Overall Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
sentiment_by_category = pd.crosstab(df["category"], df["sentiment"])
sentiment_by_category

In [4]:
sentiment_by_category.plot(kind="bar", figsize=(10,5))
plt.title("Sentiment Distribution by Category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
## Step 8: Multi-Class Text Classification
We train and compare multiple machine learning models to classify articles by category.

In [ ]:
X = df["clean_text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training size:", X_train_vec.shape)
print("Testing size:", X_test_vec.shape)

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Linear SVM": LinearSVC()
}

results = []

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)

    results.append({"Model": name, "Accuracy": acc})

    print(f"\n{name}")
    print("Accuracy:", round(acc, 4))
    print(classification_report(y_test, preds))

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
results_df

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()

In [ ]:
# Pick the best model manually if needed
best_model = LinearSVC()
best_model.fit(X_train_vec, y_train)
best_preds = best_model.predict(X_test_vec)

print("Best Model Accuracy:", accuracy_score(y_test, best_preds))
print(confusion_matrix(y_test, best_preds))

In [ ]:
## Step 9: Named Entity Recognition
We extract named entities such as PERSON, ORG, GPE, DATE, and MONEY.

In [ ]:
sample_text = df["content"].iloc[0]
doc = nlp(sample_text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

In [ ]:
entity_data = []

for text in df["content"].head(50):  # sample for speed
    doc = nlp(str(text))
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "DATE", "MONEY"]:
            entity_data.append((ent.text, ent.label_))

entity_df = pd.DataFrame(entity_data, columns=["Entity", "Label"])
entity_df.head()

In [ ]:
print(entity_df["Label"].value_counts())

entity_df["Label"].value_counts().plot(kind="bar")
plt.title("Named Entity Type Counts")
plt.xlabel("Entity Type")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
top_entities = entity_df["Entity"].value_counts().head(15)
print(top_entities)

In [ ]:
## Step 10: Test the NewsBot on a New Article
We now test the system using a new article that was not part of the training data.

In [ ]:
def newsbot_predict(article_text):
    cleaned = preprocess_text(article_text)
    vec = vectorizer.transform([cleaned])
    predicted_category = best_model.predict(vec)[0]
    sentiment = get_sentiment(article_text)

    doc = nlp(article_text)
    entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ in ["PERSON", "ORG", "GPE", "DATE", "MONEY"]]

    return {
        "predicted_category": predicted_category,
        "sentiment": sentiment,
        "entities": entities[:10]
    }

new_article = """
Apple announced a major new investment in artificial intelligence on Monday.
The company said it plans to expand hiring and open new research offices in California.
Industry experts believe the move will strengthen Apple's position against Google and Microsoft.
"""

result = newsbot_predict(new_article)
result

In [ ]:
print("Predicted Category:", result["predicted_category"])
print("Sentiment:", result["sentiment"])
print("Entities:")
for entity in result["entities"]:
    print(entity)